# CR Algorithm Test

## Useful Functions

In [14]:
import numpy as np

from pathlib import Path
import sys
parent_dir = Path.cwd().parent.resolve()
sys.path.append(str(parent_dir))

np.set_printoptions(precision=3)

In [ ]:
from numpy.linalg import norm
from numpy import sin, cos, tan, sqrt, pi
import matplotlib.pyplot as plt
from datetime import datetime
from dataclasses import dataclass
import pytz

In [16]:
mu_unicode = "\u03bc"
deg_unicode = "\u00b0"

In [17]:
def circle_points(radius, N = 100, position = None):
    p = (0,0) if position is None else position
    t = np.linspace(0, pi * 2, N, endpoint=False)
    return [radius * cos(t) + p[0], radius * sin(t) + p[1]]

def noise(cov):
    return np.random.multivariate_normal(
        mean = np.zeros(cov.shape[0]),
        cov=cov
    )

R_example = np.array([
    [0.5, 0, 0],
    [0, 0.5, 0],
    [0, 0, 0]
])
# print(f"Example noise: {noise(R_example)}\nwith R = \n{R_example}")

## Setup

In [18]:
import trajectory as T
from PlanetaryData import Earth, Luna, Sun
from Spice import SpiceImporter
from algorithms import ChristianRobinson
from Constants import RAD_TO_DEG, DEG_TO_RAD, ARCSEC_TO_RAD, RAD_TO_ARCSEC

## Camera

In [19]:
class Camera:
    def __init__(self, *, fov, f, n_pixels):
        assert len(n_pixels) == 2
        self.fov = fov
        self.f = f
        self.n_pixels = n_pixels

In [20]:
# # Camera specs 
# # e.g. https://spinworks.pt/products/star-tracker-st1/

# # Inputs
# fov_diag = 18.2 * DEG_TO_RAD
# f = 50e-3               # focal length
# n_pixels = (2048, 2048)
# mu = 5.5e-6             # pixel spacing [m]
# sigma = 2 * ARCSEC_TO_RAD / 3 # Gives in 3-sigma

# # Calculated
# mu_angle = mu / f       # pixel angle resolution [rad]
# print(f"Pixel anglular resolution: {mu_angle*RAD_TO_ARCSEC:.2f} arcseconds")
# fov = fov_diag / np.sqrt(2)
# print(f"FOV: {fov*RAD_TO_DEG:.2f}º")

In [21]:
# Camera specs 
# e.g. https://satsearch.co/products/arcsec-sagitta-star-tracker?utm_source=chatgpt.com

camera = "sagitta"

if camera == "sagitta":
    # Inputs
    fov = 25.2 * DEG_TO_RAD
    f = 2500e-3               # focal length
    n_pixels = np.array([2048, 2048])   # Assumed from previous star tracker

    cross_boresight_sigma = 2 * ARCSEC_TO_RAD
    boresight_sigma = 10 * ARCSEC_TO_RAD 
    # Calculated
    mu_angle = fov/n_pixels[0]       # pixel angle resolution [rad/px]
    mu = f * mu_angle             # pixel spacing [m/px]

    
print(f"Pixel anglular resolution: {mu_angle*RAD_TO_ARCSEC:.2f} arcs/px")
print(f"Pixel resolution: {mu*1000000:.2f} {mu_unicode}m/px")
print(f"FOV: {fov*RAD_TO_DEG:.2f}{deg_unicode}")

Pixel anglular resolution: 44.30 arcs/px
Pixel resolution: 536.89 μm/px
FOV: 25.20°


In [ ]:
@dataclass
class CameraSpecs:
    f: float
    mu: float
    n_pixels: float
    alpha: float = 0

@dataclass
class WorldSpecs:
    r_truth: np.ndarray
    a: float
    b: float
    c: float

In [28]:
def camera_specs_init(f, mu, n_pixels, alpha = 0):
    dx = dy = f / mu
    alpha = 0           # Assume no skew
    u_p, v_p = n_pixels / 2

    K_inv = np.array([
        [1/dx, -alpha/dx/dy, (alpha*v_p - dy*u_p)/dx/dy],
        [0, 1/dy, -v_p/dy],
        [0, 0, 1]
    ])

    return K_inv, u_p, v_p

K_inv, u_p, v_p = camera_specs_init(f, mu, n_pixels)

print(f"K_inv:\n{K_inv}")

K_inv:
[[ 2.148e-04  0.000e+00 -2.199e-01]
 [ 0.000e+00  2.148e-04 -2.199e-01]
 [ 0.000e+00  0.000e+00  1.000e+00]]


In [29]:
# Physical world constants
# z-axis pointed exactly along line
r_truth = np.array([0, 0, 7000])*1e3 # m


# Base
a = b = c = 3000e3

# # Earth
# a = 6378137 # km
# b = 6378137 # km
# c = 6356752 # km

In [24]:
# Generating image
N_PLANET_POINTS = 100 # Number of radial points 
planet_img_loc = (u_p, v_p) # Could change
planet_ang = np.arcsin(a / norm(r_truth)) # From center to edge[rad]
apparent_radius_px = dx * tan(planet_ang) # Just using x direction for now
body_points = circle_points(apparent_radius_px,
                            position=planet_img_loc)
u_points = np.vstack((body_points, np.ones(N_PLANET_POINTS))).T 

In [25]:
sigma = 0.5 # [px]
R = np.array([
    [sigma**2,0,0],
    [0,sigma**2,0],
    [0,0,0]
])
u_noise = np.array([v + noise(R) for v in u_points])

In [26]:
cr_alg = ChristianRobinson(K_inv, a, b, c)
T_p_c = np.eye(3)
sc_rot_angle = 90 * DEG_TO_RAD
T_p_c = np.array([
    [1, 0, 0],
    [0, cos(sc_rot_angle), sin(sc_rot_angle)],
    [0, -sin(sc_rot_angle), cos(sc_rot_angle)]
])
rc = cr_alg.run(u_noise, T_p_c)
print(f"CR Position: {rc/1000} km")
print("CR stats:")
print(f"- Absolute error: {(rc - r_truth)} km")
print(f"- Relative error: {abs(rc - r_truth)/norm(r_truth)}")
# 

CR Position: [-7.183e-02 -1.275e-01  7.000e+03] km
CR stats:
- Absolute error: [ -71.826 -127.452  -26.675] km
- Relative error: [1.026e-05 1.821e-05 3.811e-06]
